# Phần 1: Câu hỏi Trắc nghiệm

## Load data - Đọc dữ liệu

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = 'dataset/'

# Master Data
products = pd.read_csv(DATA_DIR + 'products.csv')
customers = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
promotions = pd.read_csv(DATA_DIR + 'promotions.csv', parse_dates=['start_date','end_date'])
geography = pd.read_csv(DATA_DIR + 'geography.csv')

# Transaction Data
orders = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv', low_memory=False)
payments = pd.read_csv(DATA_DIR + 'payments.csv')
shipments = pd.read_csv(DATA_DIR + 'shipments.csv', parse_dates=['ship_date','delivery_date'])
returns = pd.read_csv(DATA_DIR + 'returns.csv', parse_dates=['return_date'])
reviews = pd.read_csv(DATA_DIR + 'reviews.csv', parse_dates=['review_date'])

# Analytics Data
sales = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])
# sales_test = pd.read_csv(DATA_DIR + 'sales_test.csv')

# Operational Data
inventory = pd.read_csv(DATA_DIR + 'inventory.csv', parse_dates=['snapshot_date'])
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])

print("Data loaded successfully!")

Data loaded successfully!


## Câu 1: 
> Trong số các khách hàng có nhiều hơn một đơn hàng, **trung vị số ngày giữa hai lần mua liên tiếp** (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)
> 
> *A) 30 ngày*
> 
> *B) 90 ngày*
> 
> *C) 144 ngày*
> 
> *D) 365 ngày*

In [2]:
# Chuyển đổi cột 'order_date' sang định dạng datetime (vì có thể đang ở dạng chuỗi)
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Sắp xếp dữ liệu theo 'customer_id' và 'order_date' (để gom nhóm các đơn hàng của cùng một khách hàng và theo thứ tự thời gian)
orders = orders.sort_values(by = ['customer_id', 'order_date'])

# Lọc customer_id có nhiều hơn 1 đơn hàng (tạo cột 'prev_date' để lưu ngày của đơn hàng trước đó)
orders['prev_date'] = orders.groupby('customer_id')['order_date'].shift(1)

# Tính khoảng cách ngày giữa 2 đơn hàng liên tiếp (cột 'gap' = 'order_date' - 'prev_date')
orders['gap'] = (orders['order_date'] - orders['prev_date']).dt.days

# Loại bỏ các giá trị NaN trong cột 'gap' (các đơn hàng đầu tiên của mỗi khách hàng là NaN vì không có đơn hàng trước đó để so sánh)
gap = orders['gap'].dropna()

# Tính trung vị số ngày giữa 2 đơn hàng liên tiếp
median_gap = gap.median()

print(f"Trung vị số ngày giữa 2 đơn hàng liên tiếp: {median_gap} ngày")

Trung vị số ngày giữa 2 đơn hàng liên tiếp: 144.0 ngày


Vậy đáp án là **(C) 144 ngày**.

## Câu 2:
> Phân khúc sản phẩm (`segment`) nào trong `products.csv` có **tỷ suất lợi nhuận gộp trung bình** cao nhất? $$\dfrac{(\text{price} − \text{cogs})}{\text{price}}$$
>
> *A) Premium*
>
> *B) Performance*
>
> *C) Activewear*
>
> *D) Standard*

In [3]:
# Tính tỷ suất lợi nhuận gộp cho từng sản phẩm (gross_margin = (price - cogs) / price)
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

# Tính trung bình tỷ suất lợi nhuận gộp cho từng phân khúc sản phẩm ('segment')
segment_gross_margin = products.groupby('segment')['gross_margin'].mean()

print("Chi tiết các phân khúc sản phẩm với tỷ suất lợi nhuận gộp trung bình:")
print(segment_gross_margin.sort_values(ascending=False).to_string())

# Tìm phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất
best_segment = segment_gross_margin.idxmax()
best_margin = segment_gross_margin.max()

print(f"Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất: '{best_segment}'")
print(f"Giá trị trung bình của tỷ suất lợi nhuận gộp cho phân khúc này: {best_margin:.2%}")

Chi tiết các phân khúc sản phẩm với tỷ suất lợi nhuận gộp trung bình:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất: 'Standard'
Giá trị trung bình của tỷ suất lợi nhuận gộp cho phân khúc này: 31.34%


Vậy đáp án là **(D) Standard**.

## Câu 3:
> Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (join `returns` với `products` theo `product_id`), **lý do trả hàng** nào xuất hiện nhiều nhất?
>
> *A) defective*
>
> *B) wrong_size*
>
> *C) changed_mind*
>
> *D) not_as_described*

In [4]:
# Nối bảng 'returns' với 'products' theo 'product_id' (dùng inner join)
returns_products = pd.merge(returns, products, on = 'product_id', how = 'inner')

# Lọc các bản ghi thuộc danh mục 'Streetwear'
streetwear_returns = returns_products[returns_products['category'] == 'Streetwear']

# Đếm tần suất các lý do trả hàng
reason_counts = streetwear_returns['return_reason'].value_counts()

print("Chi tiết số lần các lý do trả hàng xuất hiện:")
print(reason_counts.sort_values(ascending=False).to_string())

# Tìm lý do trả hàng phổ biến nhất
most_common_reason = reason_counts.idxmax()
count_most_common_reason = reason_counts.max()

print(f"Lý do trả hàng phổ biến nhất cho sản phẩm Streetwear: '{most_common_reason}'")
print(f"Số lượng trả hàng với lý do này: {count_most_common_reason}")

Chi tiết số lần các lý do trả hàng xuất hiện:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Lý do trả hàng phổ biến nhất cho sản phẩm Streetwear: 'wrong_size'
Số lượng trả hàng với lý do này: 7626


Vậy đáp án là **(B) wrong_size**.

## Câu 4:
> Trong `web_traffic.csv`, nguồn truy cập (`traffic_source`) nào có **tỷ lệ thoát trung bình** (`bounce_rate`) **thấp nhất** trên tất cả các ngày xuất hiện nguồn đó trong cột `traffic_source`?
>
> *A) organic_search*
>
> *B) paid_search*
>
> *C) email_campaign*
>
> *D) social_media*

In [5]:
# Tính tỷ lệ thoát trung bình cho từng nguồn truy cập
bounce_rate_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean()

print("Chi tiết tỷ lệ thoát trung bình của các nguồn truy cập:")
print(bounce_rate_by_source.sort_values(ascending=True).to_string())

# Tìm nguồn truy cập có tỷ lệ thoát trung bình thấp nhất
best_source = bounce_rate_by_source.idxmin()
best_bounce_rate = bounce_rate_by_source.min()

print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: '{best_source}'")
print(f"Tỷ lệ thoát trung bình cho nguồn truy cập này: {best_bounce_rate:.4%}")

Chi tiết tỷ lệ thoát trung bình của các nguồn truy cập:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: 'email_campaign'
Tỷ lệ thoát trung bình cho nguồn truy cập này: 0.4458%


Vậy đáp án là **(C) email_campaign**.

## Câu 5:
> Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là `promo_id` không null) xấp xỉ là bao nhiêu?
>
> *A) 12%*
>
> *B) 25%*
>
> *C) 39%*
>
> *D) 54%*

In [6]:
# Đếm số dòng có áp dụng khuyến mãi
promo_count = order_items['promo_id'].notnull().sum()

# Đếm tổng số đơn hàng
total_orders = len(order_items)

# Tính tỷ lệ phần trăm đơn hàng có áp dụng khuyến mãi
promo_percentage = (promo_count / total_orders) * 100

print(f"Tỷ lệ phần trăm đơn hàng có áp dụng khuyến mãi: {promo_percentage:.2f}%")

Tỷ lệ phần trăm đơn hàng có áp dụng khuyến mãi: 38.66%


Vậy đáp án là **(C) 39%**.

## Câu 6:
> Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng** cao nhất? $$\dfrac{\text{tổng số đơn}}{\text{số khách hàng trong nhóm}}$$
>
> *A) 55+*
>
> *B) 25–34*
>
> *C) 35–44*
>
> *D) 45–54*

In [7]:
# Lọc các khách hàng có age_group khác null
customers_age = customers[customers['age_group'].notnull()]

# Kết hợp với bảng 'orders' để đếm số đơn hàng của từng khách hàng
age_orders = pd.merge(customers_age, orders, on = 'customer_id', how = 'inner')

# Tính số đơn hàng trung bình trên mỗi khách hàng cho từng nhóm tuổi
age_orders_mean = age_orders.groupby('age_group')['order_id'].count() / age_orders.groupby('age_group')['customer_id'].nunique()

print("Chi tiết số đơn hàng trung bình trên mỗi khách hàng của mỗi nhóm tuổi:")
print(age_orders_mean.sort_values(ascending=False).to_string())

# Tìm nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất
best_age_group = age_orders_mean.idxmax()
best_order_count = age_orders_mean.max()

print(f"Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: '{best_age_group}'")
print(f"Số đơn hàng trung bình trên mỗi khách hàng cho nhóm tuổi này: {best_order_count:.2f}")

Chi tiết số đơn hàng trung bình trên mỗi khách hàng của mỗi nhóm tuổi:
age_group
55+      7.268731
45-54    7.220264
35-44    7.206159
25-34    7.112230
18-24    7.068577
Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: '55+'
Số đơn hàng trung bình trên mỗi khách hàng cho nhóm tuổi này: 7.27


Vậy đáp án là **(A) 55+**.

## Câu 7: 
> Vùng (`region`) nào trong `geography.csv` tạo ra **tổng doanh thu cao nhất** trong `sales_train.csv`?
>
> *A) West*
>
> *B) Central*
>
> *C) East*
>
> *D) Cả ba vùng có doanh thu xấp xỉ bằng nhau*

In [8]:
# Kết hợp bảng 'orders', 'payments' với 'geography' để tính tổng doanh thu từng vùng
orders_payments_geography = pd.merge(orders, payments, on = 'order_id', how = 'inner')
orders_payments_geography = pd.merge(orders_payments_geography, geography, on = 'zip', how = 'inner')

# Tính tổng doanh thu từng vùng
revenue_by_region = orders_payments_geography.groupby('region')['payment_value'].sum()

print("Chi tiết tổng doanh thu của từng vùng:")
print(revenue_by_region.sort_values(ascending=False).to_string())

# Tìm vùng có doanh thu cao nhất
best_region = revenue_by_region.idxmax()
best_revenue = revenue_by_region.max()

print(f"Vùng có doanh thu cao nhất: '{best_region}'")
print(f"Tổng doanh thu cho vùng này: ${best_revenue:,.2f}")

Chi tiết tổng doanh thu của từng vùng:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Vùng có doanh thu cao nhất: 'East'
Tổng doanh thu cho vùng này: $7,291,150,819.12


Vậy đáp án là **(C) East**.

## Câu 8:
> Trong các đơn hàng có `order_status = ’cancelled’` trong `orders.csv`, **phương thức thanh toán** nào được sử dụng nhiều nhất?
>
> *A) credit_card*
>
> *B) cod*
>
> *C) paypal*
>
> *D) bank_transfer*

In [9]:
# Lọc các đơn hàng có tình trạng 'cancelled'
cancelled = orders[orders['order_status'] == 'cancelled']
cancelled

# Kết hợp bảng 'cancelled' với 'payments'
cancelled_payments = pd.merge(cancelled, payments, on = 'order_id', how = 'left')

# Tính tổng số lần sử dụng của từng phương thức thanh toán trong các đơn hàng bị hủy
payment_method_counts = cancelled_payments['payment_method_y'].value_counts()

print("Chi tiết số lần sử dụng của mỗi phương thức thanh toán trong các đơn hàng bị hủy:")
print(payment_method_counts.sort_values(ascending=False).to_string())

# Tìm phuơng thức thanh toán phổ biến nhất trong các đơn hàng bị hủy
most_common_payment_method = payment_method_counts.idxmax()
count_most_common_payment_method = payment_method_counts.max()

print(f"Phương thức thanh toán phổ biến nhất trong các đơn hàng bị hủy: '{most_common_payment_method}'")
print(f"Số lần sử dụng của phương thức thanh toán này trong các đơn hàng bị hủy: {count_most_common_payment_method}")

Chi tiết số lần sử dụng của mỗi phương thức thanh toán trong các đơn hàng bị hủy:
payment_method_y
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Phương thức thanh toán phổ biến nhất trong các đơn hàng bị hủy: 'credit_card'
Số lần sử dụng của phương thức thanh toán này trong các đơn hàng bị hủy: 28452


Vậy đáp án là **(A) credit_card**.

## Câu 9:

> Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có **tỷ lệ trả hàng cao nhất**, được định nghĩa là số bản ghi trong `returns` chia cho số dòng trong `order_items` (join với `products` theo `product_id`)?
>
> *A) S*
>
> *B) M*
>
> *C) L*
>
> *D) XL*

In [10]:
# Kết hợp bảng 'order_items' với 'products' để lấy thông tin kích thước của sản phẩm bán
order_items_size = pd.merge(order_items, products[['product_id', 'size']], on='product_id', how='inner')

# Đếm số bản ghi cho mỗi kích thước (S, M, L, XL)
order_items_size_counts = order_items_size['size'].value_counts()

# Kết hợp bảng 'returns' với 'products' để lấy thông tin kích thước của sản phẩm bị trả lại
returns_size = pd.merge(returns, products[['product_id', 'size']], on='product_id', how='inner')

# Đếm số bản ghi trả hàng cho mỗi kích thước
returns_size_counts = returns_size['size'].value_counts()

# Tính tỷ lệ trả hàng cho từng kích thước
return_rate = returns_size_counts.reindex(['S', 'M', 'L', 'XL']) / order_items_size_counts.reindex(['S', 'M', 'L', 'XL'])

print("Chi tiết tỷ lệ trả hàng của mỗi kích thước:")
print(return_rate.sort_values(ascending=False).to_string())

# Tìm kích thước có tỷ lệ trả hàng cao nhất
worst_size = return_rate.idxmax()
highest_rate = return_rate.max()

print(f"Kích thước có tỷ lệ trả hàng cao nhất là: '{worst_size}'")
print(f"Tỷ lệ trả hàng của kích thước này: {highest_rate:.4f}")

Chi tiết tỷ lệ trả hàng của mỗi kích thước:
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Kích thước có tỷ lệ trả hàng cao nhất là: 'S'
Tỷ lệ trả hàng của kích thước này: 0.0565


Vậy đáp án là **(A) S**.

## Câu 10: 
> Trong `payments.csv`, **kế hoạch trả góp** nào có **giá trị thanh toán trung bình trên mỗi đơn hàng** cao nhất?
>
> *A) 1 kỳ (trả một lần)*
>
> *B) 3 kỳ*
>
> *C) 6 kỳ*
>
> *D) 12 kỳ*

In [11]:
# Tính giá trị thanh toán trung bình trên mỗi đơn hàng theo từng kế hoạch trả góp
payments_installments_means = payments.groupby('installments')['payment_value'].mean()

print("Chi tiết giá trị thanh toán trung bình mỗi đơn hàng của mỗi kế hoạch trả góp:")
print(payments_installments_means.sort_values(ascending=False).to_string())

# Tìm kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất
best_installments = payments_installments_means.idxmax()
highest_installments = payments_installments_means.max()

print(f"Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: '{best_installments} kỳ'")
print(f"Giá trị thanh toán trung bình cho kế hoạch này: ${highest_installments:,.2f}")

Chi tiết giá trị thanh toán trung bình mỗi đơn hàng của mỗi kế hoạch trả góp:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất là: '6 kỳ'
Giá trị thanh toán trung bình cho kế hoạch này: $24,446.65


Vậy đáp án là **(C) 6 kỳ**.